# Reproduce the two research studies
Run from this repository. Synthetic experiments require no private data or model credentials. The operational audit aggregates cannot reconstruct the missing original source records.

## 1. Locate the package
`Path` represents a filesystem location. The loop walks upward until it finds the reproduction script, allowing this notebook to work from the notebooks folder or repository root.

In [ ]:
from pathlib import Path
import os,sys
root=Path.cwd().resolve()
while not (root/'reproduce.py').exists():
    if root.parent==root: raise FileNotFoundError('Extract the complete repository first')
    root=root.parent
os.chdir(root)
print(root)


## 2. Run the experiments
The subprocess runs the same Python interpreter as this notebook. `check=True` stops on a failed process. The script regenerates paired market runs, verifies contrasts, and runs the coefficient diagnostic. Approximately 1–3 minutes on a typical CPU; hardware affects runtime.

In [ ]:
import subprocess
subprocess.run([sys.executable,'reproduce.py'],check=True)


## 3. Read the market result
A positive participation–shock interaction means that raising adaptive capital makes the shock add more drawdown. The values below are multiplied by 100 to express percentage points. Intervals describe randomness across seeds, not market calibration uncertainty.

In [ ]:
import pandas as pd
table=pd.read_csv('results/participation_contrast_summary.csv')
selected=table[(table.shared_error==0.9)&(table.metric=='participation_shock_interaction')].copy()
selected[['mean','low95','high95']]*=100
selected[['profile','n','mean','low95','high95']]


## 4. Read the causal diagnostic
Mean squared error averages the squared gap between estimated and true synthetic coefficients. Smaller means better coefficient recovery; it is not a trading-return metric. Training regimes are oracle labels. Emission reversal changes how the latent regime is observed.

In [ ]:
pd.read_csv('results/cmtf/synthetic_summary.csv')[['scenario','method','mean_mse','mcse_mse']]


## 5. Inspect observed response agreement
These are exported-input comparisons rather than fresh model inference. Agreement measures matching direction labels and does not establish accuracy or profitability.

In [ ]:
import json
report=json.loads(Path('results/common_panel.json').read_text())
pd.DataFrame(report['common_panel_pairwise'])


## Scope
The six market profiles are assumed policies, not six actual LLMs. The CMTF router checks declared flags; it does not prove identification. The public package omits private raw logs and original prompts. See DATA_AVAILABILITY.md and the manuscripts for exact claim boundaries.